
[<img src="imagens/colab-badge.png" style="width:16%; vertical-align:middle;">](https://colab.research.google.com/github/fzampirolli/pdi-vc/blob/master/notebooks_alunos/cap03/cap03_aluno.ipynb)
[<img src="imagens/github-badge.png" style="width:20%; vertical-align:middle;">](https://github.com/fzampirolli/pdi-vc)

# 3 Operações Espaciais: Histograma, Filtragem e Convolução

Este capítulo aborda o processamento de imagens diretamente no domínio espacial. Investigamos a manipulação de histogramas para realce de contraste, além do uso de filtros locais via convolução para suavização, redução de ruído e detecção de bordas.

## 3.1 Objetivos

Ao final deste capítulo, você será capaz de:

- **Manipular intensidade e pixels:** Executar operações aritméticas e lógicas (como subtração de fundo e máscaras bit a bit) e aplicar técnicas de realce por processamento de histograma (equalização e especificação);
- **Compreender fundamentos espaciais:** Entender os conceitos de vizinhança espacial, os mecanismos de convolução/correlação e o papel dos *kernels* (máscaras);
- **Aplicar filtragem espacial de suavização:** Utilizar filtros lineares de média e Gaussiano para redução de ruído e atenuação de detalhes;
- **Aplicar filtragem espacial de realce:** Dominar o uso de filtros de nitidez e bordas, como o Laplaciano, Sobel e a técnica de *Unsharp Masking*;
- **Utilizar filtros de ordem:** Aplicar o filtro de mediana para a remoção eficaz de ruídos específicos, como o ruído do tipo sal e pimenta;
- **Resolver problemas práticos:** Combinar essas técnicas de processamento e filtragem no pré-processamento de imagens para aplicações reais.

## 3.2 Configuração do Ambiente

In [ ]:
import os, importlib, urllib.request
import numpy as np
import matplotlib.pyplot as plt
import cv2

# Baixar morph.py se necessário
BASE_URL = "https://raw.githubusercontent.com/fzampirolli/pdi-vc/master/morph"
for f in ["morph.py"]:
    if not os.path.exists(f):
        urllib.request.urlretrieve(f"{BASE_URL}/{f}", f)

import morph
importlib.reload(morph)
from morph import mm

print("✅ Ambiente pronto")

In [ ]:
# Imagem de exemplo
url = "https://upload.wikimedia.org/wikipedia/en/7/7d/Lenna_%28test_image%29.png"
img_color = mm.read(url)
img = mm.gray(img_color)
print(f"Imagem carregada: {img.shape} | Tipo: {img.dtype}")

## 3.3 Operações Aritméticas e Lógicas

Operações aritméticas e lógicas são fundamentais no processamento espacial de imagens. Elas permitem combinar imagens, remover fundos, destacar regiões de interesse e construir máscaras binárias para segmentação simples.

In [ ]:
img1 = img_gray.copy()
img2 = cv.GaussianBlur(img_gray, (21,21), 0)

# Subtração de fundo
sub = cv.absdiff(img1, img2)

# Máscara binária
_, mask = cv.threshold(sub, 25, 255, cv.THRESH_BINARY)

fig, ax = plt.subplots(1,3, figsize=(12,4))

ax[0].imshow(img1, cmap='gray')
ax[0].set_title("Original")
ax[0].axis('off')

ax[1].imshow(sub, cmap='gray')
ax[1].set_title("Subtração")
ax[1].axis('off')

ax[2].imshow(mask, cmap='gray')
ax[2].set_title("Máscara")
ax[2].axis('off')

plt.tight_layout()
plt.show()

**Figura 3.1:** Operações aritméticas e máscaras binárias.


## 3.4 Processamento de Histogramas

Histogramas descrevem a distribuição de intensidades em uma imagem. Técnicas como equalização e especificação de histogramas são utilizadas para melhorar contraste e realçar detalhes.

In [ ]:
img_eq = cv.equalizeHist(img_gray)

fig, ax = plt.subplots(2,2, figsize=(10,7))

ax[0,0].imshow(img_gray, cmap='gray')
ax[0,0].set_title("Original")
ax[0,0].axis('off')

ax[0,1].hist(img_gray.ravel(), bins=256, range=[0,256])
ax[0,1].set_title("Histograma Original")

ax[1,0].imshow(img_eq, cmap='gray')
ax[1,0].set_title("Equalizada")
ax[1,0].axis('off')

ax[1,1].hist(img_eq.ravel(), bins=256, range=[0,256])
ax[1,1].set_title("Histograma Equalizado")

plt.tight_layout()
plt.show()

**Figura 3.2:** Equalização de histograma para realce de contraste.


## 3.5 Realce por Unsharp Masking

A técnica de *Unsharp Masking* aumenta a nitidez ao reforçar componentes de alta frequência da imagem. O procedimento combina a imagem original com uma versão borrada para destacar detalhes e bordas.

In [ ]:
blur = cv.GaussianBlur(img_gray, (9,9), 2)
unsharp = cv.addWeighted(img_gray, 1.5, blur, -0.5, 0)

fig, ax = plt.subplots(1,2, figsize=(10,4))

ax[0].imshow(img_gray, cmap='gray')
ax[0].set_title("Original")
ax[0].axis('off')

ax[1].imshow(unsharp, cmap='gray')
ax[1].set_title("Unsharp Masking")
ax[1].axis('off')

plt.tight_layout()
plt.show()

**Figura 3.3:** Realce de nitidez utilizando Unsharp Masking.



## 3.6 Introdução às Operações Espaciais

Uma operação espacial modifica o valor de um pixel considerando os valores de seus vizinhos.

Essas operações normalmente utilizam uma pequena matriz chamada:

- máscara;
- kernel;
- filtro;
- janela de convolução.

A ideia central consiste em percorrer a imagem aplicando operações locais sobre regiões vizinhas.


## 3.7 Convolução Bidimensional

A convolução bidimensional é uma das operações mais importantes do PDI.

Seja:

$$
f(x,y)
$$

uma imagem de entrada e:

$$
w(s,t)
$$

um kernel de convolução.

A saída é dada por:

$$
g(x,y)=\sum_{s=-a}^{a}\sum_{t=-b}^{b} w(s,t)f(x-s,y-t)
$$

Essa operação permite implementar:
- suavização;
- realce;
- detecção de bordas;
- filtragem passa-baixa;
- filtragem passa-alta.

In [ ]:
import os
import importlib
import urllib.request

import numpy as np
import matplotlib.pyplot as plt
import cv2

# Baixar morph.py se necessário
BASE_URL = "https://raw.githubusercontent.com/fzampirolli/pdi-vc/master/"

if not os.path.exists("morph.py"):
    urllib.request.urlretrieve(BASE_URL + "notebooks/morph.py", "morph.py")

import morph as mm

plt.rcParams["figure.figsize"] = (10, 6)



## 3.8 Carregando uma imagem de exemplo

Utilizaremos uma imagem clássica do PDI para os experimentos deste capítulo.

In [ ]:
url_lena = "https://upload.wikimedia.org/wikipedia/en/7/7d/Lenna_%28test_image%29.png"

img_color = mm.read(url_lena)
img_gray = mm.gray(img_color)

print("Imagem:", img_gray.shape)


In [ ]:
plt.imshow(img_gray, cmap='gray')
plt.axis('off')
plt.show()


**Figura 3.4:** Imagem original em tons de cinza.



## 3.9 Kernels e Máscaras

Um kernel é uma pequena matriz utilizada para transformar uma imagem.

Exemplo de kernel de média:

$$
\frac{1}{9}
\begin{bmatrix}
1 & 1 & 1 \\
1 & 1 & 1 \\
1 & 1 & 1
\end{bmatrix}
$$

Esse filtro suaviza a imagem reduzindo variações locais.

In [ ]:
kernel_media = np.ones((3,3), dtype=np.float32) / 9

print(kernel_media)



## 3.10 Suavização por Média

Filtros de média reduzem ruídos ao substituir cada pixel pela média de sua vizinhança.

In [ ]:
img_media = cv2.filter2D(img_gray, -1, kernel_media)

fig, ax = plt.subplots(1,2, figsize=(12,5))

ax[0].imshow(img_gray, cmap='gray')
ax[0].set_title("Original")
ax[0].axis('off')

ax[1].imshow(img_media, cmap='gray')
ax[1].set_title("Filtro de média")
ax[1].axis('off')

plt.show()


**Figura 3.5:** Filtragem por média.



## 3.11 Filtro Gaussiano

O filtro Gaussiano produz suavização preservando melhor as estruturas da imagem.

Ele é amplamente utilizado antes da:
- detecção de bordas;
- segmentação;
- extração de características.

In [ ]:
img_gauss = cv2.GaussianBlur(img_gray, (9,9), 0)

fig, ax = plt.subplots(1,2, figsize=(12,5))

ax[0].imshow(img_gray, cmap='gray')
ax[0].set_title("Original")
ax[0].axis('off')

ax[1].imshow(img_gauss, cmap='gray')
ax[1].set_title("Gaussiano")
ax[1].axis('off')

plt.show()


**Figura 3.6:** Suavização Gaussiana.



## 3.12 Ruído em Imagens

Imagens digitais frequentemente apresentam ruídos causados por:
- sensores;
- transmissão;
- iluminação;
- compressão.

Um ruído muito comum é o ruído sal-e-pimenta.

In [ ]:
def add_salt_pepper(img, prob=0.02):

    noisy = img.copy()

    rnd = np.random.rand(*img.shape)

    noisy[rnd < prob/2] = 0
    noisy[rnd > 1 - prob/2] = 255

    return noisy

img_noise = add_salt_pepper(img_gray)


In [ ]:
plt.imshow(img_noise, cmap='gray')
plt.axis('off')
plt.show()


**Figura 3.7:** Imagem com ruído sal-e-pimenta.



## 3.13 Filtro da Mediana

O filtro da mediana é muito eficiente para remover ruído impulsivo preservando bordas.

In [ ]:
img_mediana = cv2.medianBlur(img_noise, 5)

fig, ax = plt.subplots(1,2, figsize=(12,5))

ax[0].imshow(img_noise, cmap='gray')
ax[0].set_title("Com ruído")
ax[0].axis('off')

ax[1].imshow(img_mediana, cmap='gray')
ax[1].set_title("Filtro mediana")
ax[1].axis('off')

plt.show()


**Figura 3.8:** Remoção de ruído utilizando filtro da mediana.



## 3.14 Realce e Nitidez

Filtros passa-alta enfatizam transições abruptas de intensidade, aumentando detalhes e bordas.

In [ ]:
kernel_sharp = np.array([
    [ 0, -1,  0],
    [-1,  5, -1],
    [ 0, -1,  0]
], dtype=np.float32)


In [ ]:
img_sharp = cv2.filter2D(img_gray, -1, kernel_sharp)

fig, ax = plt.subplots(1,2, figsize=(12,5))

ax[0].imshow(img_gray, cmap='gray')
ax[0].set_title("Original")
ax[0].axis('off')

ax[1].imshow(img_sharp, cmap='gray')
ax[1].set_title("Nitidez")
ax[1].axis('off')

plt.show()


**Figura 3.9:** Realce de nitidez utilizando filtro passa-alta.



## 3.15 Detecção de Bordas

Bordas representam mudanças abruptas de intensidade.

Elas são fundamentais em:
- segmentação;
- reconhecimento de objetos;
- visão computacional;
- análise estrutural.


### 3.15.1 Gradiente e Operador de Sobel

O operador de Sobel estima derivadas horizontais e verticais da imagem.

Kernel horizontal:

$$
G_x=
\\begin{bmatrix}
-1 & 0 & 1 \\\\
-2 & 0 & 2 \\\\
-1 & 0 & 1
\\end{bmatrix}
$$

Kernel vertical:

$$
G_y=
\\begin{bmatrix}
-1 & -2 & -1 \\\\
0 & 0 & 0 \\\\
1 & 2 & 1
\\end{bmatrix}
$$

In [ ]:
sobelx = cv2.Sobel(img_gray, cv2.CV_64F, 1, 0, ksize=3)
sobely = cv2.Sobel(img_gray, cv2.CV_64F, 0, 1, ksize=3)

magnitude = np.sqrt(sobelx**2 + sobely**2)

fig, ax = plt.subplots(1,3, figsize=(15,5))

ax[0].imshow(np.abs(sobelx), cmap='gray')
ax[0].set_title("Sobel X")
ax[0].axis('off')

ax[1].imshow(np.abs(sobely), cmap='gray')
ax[1].set_title("Sobel Y")
ax[1].axis('off')

ax[2].imshow(magnitude, cmap='gray')
ax[2].set_title("Magnitude")
ax[2].axis('off')

plt.show()


**Figura 3.10:** Detecção de bordas utilizando Sobel.



## 3.16 Detector de Bordas de Canny

O detector de Canny é um dos métodos mais robustos para detecção de bordas.

Etapas:
1. suavização Gaussiana;
2. cálculo do gradiente;
3. supressão de não máximos;
4. limiarização por histerese.

In [ ]:
edges = cv2.Canny(img_gray, 100, 200)

plt.imshow(edges, cmap='gray')
plt.axis('off')
plt.show()


**Figura 3.11:** Detecção de bordas utilizando Canny.



## 3.17 Comparação entre Filtros

Cada filtro possui características específicas:
- média: simples e rápido;
- Gaussiano: suavização natural;
- mediana: robusto para ruído impulsivo;
- passa-alta: realce de detalhes.

## 3.18 Resumo

Neste capítulo, exploramos operações espaciais fundamentais para o processamento digital de imagens. Estudamos técnicas de manipulação de intensidade, processamento de histogramas, convolução, filtragem linear, redução de ruído e realce de bordas.

Também analisamos filtros clássicos, como média, Gaussiano, mediana, Sobel, Laplaciano e *Unsharp Masking*, aplicando-os em tarefas práticas de pré-processamento e melhoria de imagens.


## 3.19 Uso do NotebookLM como Tutor Complementar

Sugestões de estudo:
- peça explicações sobre convolução;
- solicite exemplos de kernels;
- compare filtros lineares e não lineares;
- revise Sobel e Canny;
- explore aplicações práticas em visão computacional.


## 3.20 Lista de Exercícios

1. Implementar convolução manualmente utilizando NumPy;
2. Comparar média e Gaussiano;
3. Testar diferentes tamanhos de kernel;
4. Adicionar ruídos artificiais;
5. Comparar filtros para remoção de ruído;
6. Implementar Sobel sem OpenCV;
7. Detectar bordas em imagens reais;
8. Comparar Sobel e Canny.


## 3.21 Parte Prática com Exercícios de Programação

### 3.21.1 Objetivo deste Caderno

Os exercícios a seguir consolidam os conceitos estudados neste capítulo através de implementações práticas utilizando Python.

### 3.21.2 EP03_01 — Convolução Manual 2D

In [ ]:

# Escreva sua solução aqui



### 3.21.3 EP03_02 — Filtro da Média

In [ ]:

# Escreva sua solução aqui



### 3.21.4 EP03_03 — Filtro Gaussiano

In [ ]:

# Escreva sua solução aqui



### 3.21.5 EP03_04 — Ruído Sal-e-Pimenta

In [ ]:

# Escreva sua solução aqui



### 3.21.6 EP03_05 — Filtro da Mediana

In [ ]:

# Escreva sua solução aqui



### 3.21.7 EP03_06 — Kernel de Nitidez

In [ ]:

# Escreva sua solução aqui



### 3.21.8 EP03_07 — Detector de Bordas Sobel

In [ ]:

# Escreva sua solução aqui



### 3.21.9 EP03_08 — Magnitude do Gradiente

In [ ]:

# Escreva sua solução aqui



### 3.21.10 EP03_09 — Detector de Bordas Canny

In [ ]:

# Escreva sua solução aqui



### 3.21.11 EP03_10 — Comparação entre Filtros

In [ ]:

# Escreva sua solução aqui

import numpy as np
print(np.zeros((5,6)))

In [ ]:
url = "https://upload.wikimedia.org/wikipedia/en/7/7d/Lenna_%28test_imageikimedia.org/wikipedia/en/7/7d/Lenna_%28test_image%29.png"

print(url)